In [0]:
# Databricks notebook source

In [0]:
months_to_process = [
    ("2024", "02"), ("2024", "03"), ("2024", "04"), ("2024", "05"),
    ("2024", "06"), ("2024", "07"), ("2024", "08"), ("2024", "09"),
    ("2024", "10"), ("2024", "11"), ("2024", "12"),
]
 
NOTEBOOK_TIMEOUT_SECONDS = 3600  # 1 hora por notebook, generoso para o volume mensal
 

In [0]:
successful_months = []
failed_months = []
 
for year, month in months_to_process:
    year_month = f"{year}-{month}"
    print(f"\n=== Processando {year_month} ===")
 
    try:
        bronze_result = dbutils.notebook.run(
            "./01_bronze_ingestion",
            NOTEBOOK_TIMEOUT_SECONDS,
            {"year": year, "month": month},
        )
        print(f"Bronze [{year_month}]: {bronze_result}")
 
        if bronze_result and ("interrompida" in bronze_result or "FAILED" in bronze_result):
            print(f"Bronze falhou para {year_month}, pulando Silver deste mês.")
            failed_months.append((year_month, "bronze", bronze_result))
            continue
 
        silver_result = dbutils.notebook.run(
            "./02_silver_transformation",
            NOTEBOOK_TIMEOUT_SECONDS,
            {"year": year, "month": month},
        )
        print(f"Silver [{year_month}]: {silver_result}")
 
        successful_months.append(year_month)
 
    except Exception as e:
        print(f"Erro inesperado processando {year_month}: {e}")
        failed_months.append((year_month, "exception", str(e)))
        continue
 

## Resumo do backfill

In [0]:
print(f"Meses processados com sucesso: {len(successful_months)}")
print(successful_months)
print(f"\nMeses com falha: {len(failed_months)}")
for year_month, stage, detail in failed_months:
    print(f"  {year_month} — falhou em {stage}: {detail}")

## Execução única do Gold

Só roda se pelo menos um mês novo foi processado com sucesso — evita recomputar a Gold à toa se o
backfill inteiro falhou.

In [0]:
if successful_months:
    gold_result = dbutils.notebook.run("./03_gold_tip_behavior", NOTEBOOK_TIMEOUT_SECONDS)
    print(f"Gold recomputado: {gold_result}")
else:
    print("Nenhum mês novo processado com sucesso — Gold não foi recomputado.")